### Imports

In [ ]:
import os
import time
import dill
import json
import string
import itertools
from tqdm import trange
from pathlib import Path

import torch
import numpy as np
import pandas as pd

import sys
sys.path.append("..")

from simulation.simulation_tools import get_optimal_sim_XYP
from utils import print_time_slices

from cdt.metrics import SHD

import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng()

def prind(di): print(json.dumps(di, sort_keys=False, indent=4))

COL_NAMES = list(string.ascii_uppercase) + ["".join(a) for a in list(itertools.permutations(list(string.ascii_uppercase), r=2))]

### Data

In [ ]:
# data path
data_folder = 'variables'
folder_size = 'medium'
data_postfix = f'synthetic/{data_folder}/{folder_size}'
data_path = list(Path(".").resolve().parents)[1] / 'data' / data_postfix

# data pairs
dense = []
data_pairs = []
for fn in os.listdir(data_path / "data")[:]:
    true_data = pd.read_csv(data_path / "data" / fn)
    true_graph = torch.load(data_path / "structure" / fn.replace("ts", "struct").replace("csv", "pt"))
    data_pairs.append([true_data, true_graph])
    dense.append(true_graph.sum().item())
    # # inspect
    # print(f"- {fn}")
    # print(f"    - {true_data.shape}, {true_graph.shape}, {true_graph.sum()}")
    # print()
print(f"Average density: {np.mean(dense) / (true_graph.shape[0] * (true_graph.shape[0] - 1)):.2f} ({round(np.mean(dense), 0)} edges on average)")

### Method

#### Run

In [ ]:
folders = ['variables', 'samples', 'density', 'lags']
sizes = ['small', 'medium', 'large']
batch = 'batch_2'

archive_folder = "archive_newer_full"

for data_folder in folders:
    for folder_size in sizes:

        # data path
        print(f"\n -- {data_folder} - {folder_size} -- \n")
        data_postfix = f'{batch}/{data_folder}/{folder_size}'
        data_path = list(Path(".").resolve().parents)[1] / 'data' / 'synthetic' / data_postfix

        # data pairs
        data_pairs = []
        for fn in os.listdir(data_path / "data")[:]:
            true_data = pd.read_csv(data_path / "data" / fn)
            true_graph = torch.load(data_path / "structure" / fn.replace("ts", "struct").replace("csv", "pt"))
            data_pairs.append([true_data, true_graph])

        # archive folder
        exp_archive_path = Path(".").resolve().parents[1] / "data" / "results" / "archive" / archive_folder / data_folder / folder_size
        os.makedirs(exp_archive_path, exist_ok=True)

        # hyperparameters
        runs = 1

        # placeholders
        # results = pd.DataFrame(index=range(len(data_pairs)), columns=["ACT"])

        # data loop
        for idx, (true_data, label_graph) in enumerate(data_pairs[:]):
            # archive folder for grouped runs
            runs_archive_path = exp_archive_path / f"runs_{idx}"
            os.makedirs(runs_archive_path, exist_ok=True)
            # runs loop
            for idy in trange(runs, desc=f"idx {idx}"):
                true_graph = label_graph.clone()

                try:
                    res = get_optimal_sim_XYP(
                        true_data=true_data, 
                        sparsity_penalty=True,
                        archive_path=runs_archive_path / f"act_{idx}_run_{idy}.dill",
                    )
                    pred_graph = res['optimal_scm'].causal_structure.causal_structure_cp
                    if  true_graph.shape[2]>pred_graph.shape[2]:
                        pred_graph = torch.nn.functional.pad(input=pred_graph, pad=(0, true_graph.shape[2] - pred_graph.shape[2], 0, 0, 0, 0), value=0)
                    if  pred_graph.shape[2]>true_graph.shape[2]:
                        true_graph = torch.nn.functional.pad(input=true_graph, pad=(0, pred_graph.shape[2] - true_graph.shape[2], 0, 0, 0, 0), value=0)
                    # shd = SHD(target=true_graph.numpy(), pred=pred_graph.numpy())
                    # results.loc[idx, "ACT"] = shd
                    print("\n ------------------------------------------------------------------------------------------ \n")
                except Exception as e:
                    print(f"ACT failed for idx {idx}, run {idy}: {e}")
                    # results.loc[idx, "ACT"] = np.nan
                    continue

#### Special Cases

##### Re-run Density

In [2]:
folders = ['density']
sizes = ['small', 'medium', 'large']
batch = 'batch_density_special'

archive_folder = "archive_density_special"

for data_folder in folders:
    for folder_size in sizes:

        # data path
        print(f"\n -- {data_folder} - {folder_size} -- \n")
        data_postfix = f'{batch}/{data_folder}/{folder_size}'
        data_path = list(Path(".").resolve().parents)[1] / 'data' / 'synthetic' / data_postfix

        # data pairs
        data_pairs = []
        for fn in os.listdir(data_path / "data")[:]:
            true_data = pd.read_csv(data_path / "data" / fn)
            true_graph = torch.load(data_path / "structure" / fn.replace("ts", "struct").replace("csv", "pt"))
            data_pairs.append([true_data, true_graph])

        # archive folder
        exp_archive_path = Path(".").resolve().parents[1] / "data" / "results" / "archive" / archive_folder / data_folder / folder_size
        os.makedirs(exp_archive_path, exist_ok=True)

        # hyperparameters
        runs = 1

        # placeholders
        # results = pd.DataFrame(index=range(len(data_pairs)), columns=["ACT"])

        # data loop
        for idx, (true_data, label_graph) in enumerate(data_pairs[:]):
            # archive folder for grouped runs
            runs_archive_path = exp_archive_path / f"runs_{idx}"
            os.makedirs(runs_archive_path, exist_ok=True)
            # runs loop
            for idy in trange(runs, desc=f"idx {idx}"):
                true_graph = label_graph.clone()

                try:
                    res = get_optimal_sim_XYP(
                        true_data=true_data, 
                        archive_path=runs_archive_path / f"act_{idx}_run_{idy}.dill",
                    )
                    pred_graph = res['optimal_scm'].causal_structure.causal_structure_cp
                    if  true_graph.shape[2]>pred_graph.shape[2]:
                        pred_graph = torch.nn.functional.pad(input=pred_graph, pad=(0, true_graph.shape[2] - pred_graph.shape[2], 0, 0, 0, 0), value=0)
                    if  pred_graph.shape[2]>true_graph.shape[2]:
                        true_graph = torch.nn.functional.pad(input=true_graph, pad=(0, pred_graph.shape[2] - true_graph.shape[2], 0, 0, 0, 0), value=0)
                    # shd = SHD(target=true_graph.numpy(), pred=pred_graph.numpy())
                    # results.loc[idx, "ACT"] = shd
                    print("\n ------------------------------------------------------------------------------------------ \n")
                except Exception as e:
                    print(f"ACT failed for idx {idx}, run {idy}: {e}")
                    # results.loc[idx, "ACT"] = np.nan
                    continue


 -- density - small -- 



idx 0:   0%|          | 0/1 [00:00<?, ?it/s]

LOG: Optimal Simulation: 24 TCS configurations are to be tested ...
LOG: Optimal Simulation: archive saved at C:\Users\skypl\Documents\GitHub\TCS\data\results\archive\archive_density_special\density\small\runs_0\act_0_run_0.dill ...


idx 0: 100%|██████████| 1/1 [02:06<00:00, 126.13s/it]



 ------------------------------------------------------------------------------------------ 



...


 ------------------------------------------------------------------------------------------ 



idx 9:   0%|          | 0/1 [00:00<?, ?it/s]

LOG: Optimal Simulation: 24 TCS configurations are to be tested ...
LOG: Optimal Simulation: archive saved at C:\Users\skypl\Documents\GitHub\TCS\data\results\archive\archive_density_special\density\large\runs_9\act_9_run_0.dill ...


idx 9: 100%|██████████| 1/1 [06:15<00:00, 375.23s/it]


 ------------------------------------------------------------------------------------------ 



##### Run LCMs

In [ ]:
# # folders = ['variables', 'samples', 'density', 'lags']
# folders = ['samples', 'density', 'lags']
# sizes = ['small', 'medium', 'large']

# for data_folder in folders:
#     for folder_size in sizes:

#         # data path
#         print(f"\n -- {data_folder} - {folder_size} -- \n")
#         data_postfix = f'synthetic/{data_folder}/{folder_size}'
#         data_path = list(Path(".").resolve().parents)[1] / 'data' / data_postfix

#         # data pairs
#         dense = []
#         data_pairs = []
#         for fn in os.listdir(data_path / "data")[:]:
#             true_data = pd.read_csv(data_path / "data" / fn)
#             true_graph = torch.load(data_path / "structure" / fn.replace("ts", "struct").replace("csv", "pt"))
#             data_pairs.append([true_data, true_graph])
#             dense.append(true_graph.sum().item())

#         # archive folder
#         exp_archive_path = Path(".").resolve().parents[1] / "data" / "results" / "archive" / "with_lcm" / data_folder / folder_size
#         os.makedirs(exp_archive_path, exist_ok=True)

#         # hyperparameters
#         runs = 1

#         # placeholders
#         results = pd.DataFrame(index=range(len(data_pairs)), columns=["ACT"])

#         # data loop
#         for idx, (true_data, label_graph) in enumerate(data_pairs[:]):
#             # archive folder for grouped runs
#             runs_archive_path = exp_archive_path / f"runs_{idx}"
#             os.makedirs(runs_archive_path, exist_ok=True)
#             # runs loop
#             for idy in trange(runs, desc=f"idx {idx}"):
#                 true_graph = label_graph.clone()

#                 try:
#                     res = get_optimal_sim_XYP(
#                         true_data=true_data, 
#                         archive_path=runs_archive_path / f"act_{idx}_run_{idy}.dill",
#                     )
#                     pred_graph = res['optimal_scm'].causal_structure.causal_structure_cp
#                     if  true_graph.shape[2]>pred_graph.shape[2]:
#                         pred_graph = torch.nn.functional.pad(input=pred_graph, pad=(0, true_graph.shape[2] - pred_graph.shape[2], 0, 0, 0, 0), value=0)
#                     if  pred_graph.shape[2]>true_graph.shape[2]:
#                         true_graph = torch.nn.functional.pad(input=true_graph, pad=(0, pred_graph.shape[2] - true_graph.shape[2], 0, 0, 0, 0), value=0)
#                     shd = SHD(target=true_graph.numpy(), pred=pred_graph.numpy())
#                     results.loc[idx, "ACT"] = shd
#                     print("\n ------------------------------------------------------------------------------------------ \n")
#                 except Exception as e:
#                     print(f"ACT failed for idx {idx}, run {idy} with error: {e}")
#                     # print(f"ACT failed for idx {idx}, run {idy}")
#                     results.loc[idx, "ACT"] = np.nan
#                     print("\n ------------------------------------------------------------------------------------------ \n")
#                     continue

#         print("\n ------------------------------------------------------------------------------------------ \n")

##### Assimilate LCM runs

In [ ]:
# # space
# folders = ['variables', 'samples', 'density', 'lags']
# sizes = ['small', 'medium', 'large']

# for i, data_folder in enumerate(folders[:]):
#     for j, folder_size in enumerate(sizes):
#         print(f"\n -- {data_folder} - {folder_size} -- \n")

#         # data path
#         exp_archive_path = Path(".").resolve().parents[1] / "data" / "results" / "archive" / data_folder / folder_size
#         lcm_archive_path = Path(".").resolve().parents[1] / "data" / "results" / "archive" / "with_lcm" / data_folder / folder_size
#         data_path = list(Path(".").resolve().parents)[1] / 'data' / 'synthetic' / data_folder / folder_size

#         # data pairs
#         data_n_arch = []
#         for ind, fn in enumerate(os.listdir(data_path / "data")[:]):
#             true_data = pd.read_csv(data_path / "data" / fn)
#             true_graph = torch.load(data_path / "structure" / fn.replace("ts", "struct").replace("csv", "pt"))
#             data_n_arch.append([true_data, true_graph, exp_archive_path / f"runs_{ind}", lcm_archive_path / f"runs_{ind}"])

#         # index based on dataset
#         print(f"- exp archive datasets: {len(os.listdir(exp_archive_path))} files")
#         print(f"- lcm archive datasets: {len(os.listdir(lcm_archive_path))} files")
#         for ind, (true_data, label_graph, arch_path, lcm_arch_path) in enumerate(data_n_arch[:]):

#             print(f"\n    -- idx {ind} -- ")
#             print(f"    - exp archive repetitions: {len(os.listdir(arch_path))} files")
#             print(f"    - lcm archive repetitions: {len(os.listdir(lcm_arch_path))} files")

#             if len(os.listdir(lcm_arch_path))==0 and len(os.listdir(arch_path))==0:
#                 print(f"    - no archive found for idx={ind}, skipping this index ...")
#                 continue
            
#             if len(os.listdir(arch_path))>0:
#                 for fn in os.listdir(arch_path)[:]:
#                     # load exp archive
#                     with open(arch_path / fn, "rb") as f:
#                         exp_archive = dill.load(f)
#             else:
#                 exp_archive = {k: [] for k in exp_archive.keys()}  # please validate that the first one exists, in my case it does
            
#             if len(os.listdir(lcm_arch_path))>0:
#                 for fn in os.listdir(lcm_arch_path)[:]:
#                     # load exp archive
#                     with open(lcm_arch_path / fn, "rb") as f:
#                         lcm_archive = dill.load(f)
#             else:
#                 lcm_archive = {k: [] for k in exp_archive.keys()}

#             print(f"    - exp archive length before: {len(list(exp_archive.values())[0])} files")
#             print(f"    - lcm archive length before: {len(list(lcm_archive.values())[0])} files")

#             lcm_archive_merged = {k: exp_archive[k] + lcm_archive[k] if k!="idx" else 0 for k in exp_archive.keys()}

#             from utils import _from_full_to_cp
#             from simulation.simulation_tools import equivalence_test
#             def apply_sparsity_penalty(fit_scm_list, auc_list, probs_list, labels_list):
#                 # derive the cp-adjacency matrix
#                 cp_list = [_from_full_to_cp(x).sum().item() if isinstance(x, pd.DataFrame) 
#                                 else x.causal_structure.causal_structure_cp.sum().item() for x in fit_scm_list]
#                 # list of indices with optimal performance
#                 idx = np.argmin([np.abs(0.5-x) for x in auc_list])
#                 eq_list = [idx]
#                 print(f"\nLOG: Optimal Simulation: Sparsity Penalty: optimal case before: idx={idx} | auc={auc_list[idx]} | edges={cp_list[idx]}")
#                 # permutation loop
#                 for i, _ in enumerate(auc_list): 
#                     if i!=idx:
#                         dcs = equivalence_test(
#                             auc_o=auc_list[idx], 
#                             probs_o=probs_list[idx],
#                             labels_o=labels_list[idx],
#                             auc_i=auc_list[i], 
#                             probs_i=probs_list[i],
#                             labels_i=labels_list[i],
#                             a=0.05,
#                             n=100
#                         )
#                         if dcs:
#                             eq_list.append(i)
#                             print(f"LOG: Optimal Simulation: Sparsity Penalty: eq. found: idx={i} | auc={auc_list[i]} | edges={cp_list[i]}")
#                 eq_cp = [(i, cp_list[i]) for i in eq_list]
#                 idx = sorted(eq_cp, key=lambda x: x[1])[0][0]
#                 print(f"LOG: Optimal Simulation: Sparsity Penalty: optimal case after: idx={idx} | auc={auc_list[idx]} | edges={cp_list[idx]}\n")
#                 return idx
            
#             idx = apply_sparsity_penalty(
#                 fit_scm_list=lcm_archive_merged["fit_scm_list"], 
#                 auc_list=lcm_archive_merged["auc_list"], 
#                 probs_list=lcm_archive_merged["probs_list"], 
#                 labels_list=lcm_archive_merged["labels_list"]
#             )
#             print(f"idx : {idx}")
#             lcm_archive_merged["idx"] = idx

#             print(f"    - exp archive length after: {len(list(exp_archive.values())[0])} files")
#             print(f"    - lcm archive length after: {len(list(lcm_archive_merged.values())[0])} files")

#             # with open(lcm_arch_path / fn, "wb") as f:
#             #     dill.dump(lcm_archive_merged, f)
#             print(f"LOG: Optimal Simulation: archive saved at {lcm_arch_path / fn} ... \n")

### Check Archive

In [ ]:
# data_folder = 'variables'
# folder_size = 'medium'
# exp_archive_path = Path(".").resolve().parents[1] / "data" / "results" / "archive" / data_folder / folder_size
# data_path = list(Path(".").resolve().parents)[1] / 'data' / 'synthetic' / data_folder / folder_size

# # data pairs
# data_n_arch = []
# for ind, fn in enumerate(os.listdir(data_path / "data")[:]):
#     true_data = pd.read_csv(data_path / "data" / fn)
#     true_graph = torch.load(data_path / "structure" / fn.replace("ts", "struct").replace("csv", "pt"))
#     data_n_arch.append([true_data, true_graph, exp_archive_path / f"runs_{ind}"])

# for ind, (true_data, label_graph, arch_path) in enumerate(data_n_arch[:]):
#     print(f"\n--- {ind} ---")
#     tpe = label_graph.shape[0]**2 * (label_graph.shape[2])
#     print(f"Graph shape: {label_graph.numpy().shape} | Total possible edges: {tpe}")
    
#     shd_avg = []
#     nshd_avg = []
#     for fn in os.listdir(arch_path)[:]:
#         true_graph = label_graph.clone()
#         # load archive
#         with open(arch_path / fn, "rb") as f:
#             archive = dill.load(f)
#         idx = archive['idx']
#         # compute SHD
#         pred_graph = archive['fit_scm_list'][idx].causal_structure.causal_structure_cp
#         if  true_graph.shape[2]>pred_graph.shape[2]:
#             pred_graph = torch.nn.functional.pad(input=pred_graph, pad=(true_graph.shape[2] - pred_graph.shape[2], 0, 0, 0, 0, 0), value=0)
#         if  pred_graph.shape[2]>true_graph.shape[2]:
#             true_graph = torch.nn.functional.pad(input=true_graph, pad=(pred_graph.shape[2] - true_graph.shape[2], 0, 0, 0, 0, 0), value=0)
#         shd = SHD(target=true_graph.numpy(), pred=pred_graph.numpy())
#         # nshd = round(shd / true_graph.shape[0], 2)
#         nshd = round(shd / tpe, 2)
#         shd_avg.append(shd)
#         nshd_avg.append(nshd)
#         # print statistics
#         print(f"    - {fn}: idx={archive['idx']}, shd={shd}, nshd={nshd}, w/ len={len(archive['fit_scm_list'])}")
#     print(f"-average results :  avg_shd={np.mean(shd_avg).round(2)}, avg_nshd={np.mean(nshd_avg).round(2)}")